# Retail Dataset (v2) — Data Cleaning Notebook

This is a **new upload** with five files, some overlapping the previous dataset in name but different in content, plus one new file. We clean each one **separately** and keep them in their own tables throughout — same approach as before.

| File | What it contains | Grain (one row =) |
|---|---|---|
| `sales_daily.csv` | Daily sales, demand, price, discount, promotion | 1 store + 1 product + 1 day |
| `inventory_snapshots.csv` | Daily inventory levels, units ordered, competitor pricing | 1 store + 1 product + 1 day |
| `sku_master.csv` | Product/store → category + region lookup | 1 store + 1 product |
| `calender.csv` | Calendar attributes + seasonality + epidemic flag | 1 day |
| `sales_data.csv` | **New file** — looks like a fully combined version of the other four (same rows, every column in one place) | 1 store + 1 product + 1 day |

**Why clean `sales_data.csv` too, even though it looks redundant:** it's a separate file the client gave us, so it needs its own independent quality check — we shouldn't assume it's a perfect copy of the others until we verify that ourselves (Step 6 checks this directly).

Same method as before: for every step — *Check → Explain → Fix → Verify*.


In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print("pandas version:", pd.__version__)


pandas version: 3.0.2


## Step 1 — Load the raw data


In [2]:
sales_raw = pd.read_csv('../Original Dataset/sales_daily.csv')
inventory_raw = pd.read_csv('../Original Dataset/inventory_snapshots.csv')
sku_raw = pd.read_csv('../Original Dataset/sku_master.csv')
calendar_raw = pd.read_csv('../Original Dataset/calender.csv')
sales_data_raw = pd.read_csv('../Original Dataset/sales_data.csv')

sales = sales_raw.copy()
inventory = inventory_raw.copy()
sku = sku_raw.copy()
calendar = calendar_raw.copy()
sales_data = sales_data_raw.copy()

for name, df in [('sales', sales), ('inventory', inventory), ('sku_master', sku),
                  ('calendar', calendar), ('sales_data', sales_data)]:
    print(f"{name:15s} {df.shape}")


sales           (76000, 8)
inventory       (76000, 6)
sku_master      (100, 4)
calendar        (760, 12)
sales_data      (76000, 16)


## Step 2 — First look at each table


In [3]:
for name, df in [('SALES', sales), ('INVENTORY', inventory), ('SKU MASTER', sku),
                  ('CALENDAR', calendar), ('SALES_DATA', sales_data)]:
    print(f"\n{'='*15} {name} {'='*15}")
    print(df.dtypes)
    display(df.head(3))



=============== SALES ===============
Date              str
Store ID          str
Product ID        str
Units Sold      int64
Demand          int64
Price         float64
Discount        int64
Promotion       int64
dtype: object


,Date,Store ID,Product ID,Units Sold,Demand,Price,Discount,Promotion
0,2022-01-01,S001,P0001,102,115,72.72,5,0
1,2022-01-01,S001,P0002,117,229,80.16,15,1
2,2022-01-01,S001,P0003,114,157,62.94,10,1



=============== INVENTORY ===============
Date                      str
Store ID                  str
Product ID                str
Inventory Level         int64
Units Ordered           int64
Competitor Pricing    float64
dtype: object


,Date,Store ID,Product ID,Inventory Level,Units Ordered,Competitor Pricing
0,2022-01-01,S001,P0001,195,252,85.73
1,2022-01-01,S001,P0002,117,249,92.02
2,2022-01-01,S001,P0003,247,612,60.08



=============== SKU MASTER ===============
Store ID      str
Product ID    str
Category      str
Region        str
dtype: object


,Store ID,Product ID,Category,Region
0,S001,P0001,Electronics,North
1,S001,P0002,Clothing,North
2,S001,P0003,Clothing,North



=============== CALENDAR ===============
Date             str
Year           int64
Quarter        int64
Month          int64
MonthName        str
Day            int64
DayOfWeek      int64
DayName          str
WeekOfYear     int64
IsWeekend       bool
Seasonality      str
Epidemic       int64
dtype: object


,Date,Year,Quarter,Month,MonthName,Day,DayOfWeek,DayName,WeekOfYear,IsWeekend,Seasonality,Epidemic
0,2022-01-01,2022,1,1,January,1,6,Saturday,52,True,Winter,0
1,2022-01-02,2022,1,1,January,2,7,Sunday,52,True,Winter,0
2,2022-01-03,2022,1,1,January,3,1,Monday,1,False,Winter,0



=============== SALES_DATA ===============
Date                      str
Store ID                  str
Product ID                str
Category                  str
Region                    str
Inventory Level         int64
Units Sold              int64
Units Ordered           int64
Price                 float64
Discount                int64
Weather Condition         str
Promotion               int64
Competitor Pricing    float64
Seasonality               str
Epidemic                int64
Demand                  int64
dtype: object


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand
0,1/1/2022,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115
1,1/1/2022,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229
2,1/1/2022,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157


**Notice:** `sales_daily`, `inventory_snapshots`, and `calender` are much like before — but this version adds new columns: `Demand`, `Promotion`, `Seasonality`, `Epidemic`. `sku_master` is now keyed by **Store ID + Product ID** (100 rows = 5 stores × 20 products), not just Product ID as before. `sales_data` bundles everything into one wide table. As before, `Date` is text everywhere.


## Step 3 — Check for missing values


In [4]:
for name, df in [('sales', sales), ('inventory', inventory), ('sku', sku),
                  ('calendar', calendar), ('sales_data', sales_data)]:
    print(f"{name}: {df.isnull().sum().sum()} missing values")


sales: 0 missing values
inventory: 0 missing values
sku: 0 missing values
calendar: 0 missing values
sales_data: 0 missing values


No missing values anywhere. Confirmed, not assumed.


## Step 4 — Check for duplicate rows


In [5]:
print("Exact duplicate rows:")
for name, df in [('sales', sales), ('inventory', inventory), ('sku', sku),
                  ('calendar', calendar), ('sales_data', sales_data)]:
    print(f"  {name}: {df.duplicated().sum()}")

print("\nDuplicate business keys (Date + Store ID + Product ID):")
print("  sales:      ", sales.duplicated(subset=['Date', 'Store ID', 'Product ID']).sum())
print("  inventory:  ", inventory.duplicated(subset=['Date', 'Store ID', 'Product ID']).sum())
print("  sales_data: ", sales_data.duplicated(subset=['Date', 'Store ID', 'Product ID']).sum())

print("\nDuplicate sku_master keys (Store ID + Product ID):")
print("  sku:        ", sku.duplicated(subset=['Store ID', 'Product ID']).sum())

print("\nDuplicate calendar dates:")
print("  calendar:   ", calendar.duplicated(subset=['Date']).sum())


Exact duplicate rows:
  sales: 0
  inventory: 0
  sku: 0
  calendar: 0


  sales_data: 0

Duplicate business keys (Date + Store ID + Product ID):


  sales:       0


  inventory:   0
  sales_data:  0

Duplicate sku_master keys (Store ID + Product ID):
  sku:         0

Duplicate calendar dates:
  calendar:    0


No duplicates of any kind.


## Step 5 — Fix date columns (format mismatch, again)

Same issue as the previous dataset: `sales`, `inventory`, and `calendar` use `YYYY-MM-DD`, but `sales_data` uses `M/D/YYYY`. We convert every `Date` column to a real `datetime64` using the correct format for each file.


In [6]:
sales['Date'] = pd.to_datetime(sales['Date'], format='%Y-%m-%d')
inventory['Date'] = pd.to_datetime(inventory['Date'], format='%Y-%m-%d')
calendar['Date'] = pd.to_datetime(calendar['Date'], format='%Y-%m-%d')
sales_data['Date'] = pd.to_datetime(sales_data['Date'], format='%m/%d/%Y')

for name, df in [('sales', sales), ('inventory', inventory), ('calendar', calendar), ('sales_data', sales_data)]:
    print(f"{name:12s} dtype={df['Date'].dtype}  range={df['Date'].min().date()} -> {df['Date'].max().date()}")


sales        dtype=datetime64[us]  range=2022-01-01 -> 2024-01-30
inventory    dtype=datetime64[us]  range=2022-01-01 -> 2024-01-30
calendar     dtype=datetime64[us]  range=2022-01-01 -> 2024-01-30
sales_data   dtype=datetime64[us]  range=2022-01-01 -> 2024-01-30


In [7]:
full_range = pd.date_range(sales['Date'].min(), sales['Date'].max(), freq='D')
missing_days = set(full_range) - set(calendar['Date'])
print("Expected days:", len(full_range), " | Calendar rows:", len(calendar), " | Missing days:", len(missing_days))


Expected days: 760  | Calendar rows: 760  | Missing days: 0


All date ranges align (`2022-01-01` to `2024-01-30`), and the calendar has zero gaps — one row per day across the whole span.


## Step 6 — Referential integrity: do IDs match, and does `sales_data` actually agree with the other tables?

Two things to check here:
1. Do Store/Product IDs match across `sales`, `inventory`, and `sku_master`?
2. Since `sales_data` claims to bundle everything together, does its `Category`/`Region` actually match what `sku_master` says for the same store + product? This is the real test of whether it's a trustworthy combined file.


In [8]:
sales_products, inv_products, sku_products = set(sales['Product ID']), set(inventory['Product ID']), set(sku['Product ID'])
sales_stores, inv_stores = set(sales['Store ID']), set(inventory['Store ID'])

print("Products match sales <-> inventory:", sales_products == inv_products)
print("Products match sales <-> sku_master:", sales_products == sku_products)
print("Stores match sales <-> inventory:", sales_stores == inv_stores)
print(f"Distinct products: {len(sales_products)}  |  Distinct stores: {len(sales_stores)}")


Products match sales <-> inventory: True
Products match sales <-> sku_master: True
Stores match sales <-> inventory: True
Distinct products: 20  |  Distinct stores: 5


In [9]:
# Compare sales_data's own Category/Region against sku_master, for every store+product pair
check = (sales_data[['Store ID', 'Product ID', 'Category', 'Region']]
         .drop_duplicates()
         .merge(sku, on=['Store ID', 'Product ID'], suffixes=('_salesdata', '_skumaster'), how='outer', indicator=True))

print("Merge match status:\n", check['_merge'].value_counts())

category_mismatch = (check['Category_salesdata'] != check['Category_skumaster']).sum()
region_mismatch = (check['Region_salesdata'] != check['Region_skumaster']).sum()
print(f"\nCategory mismatches: {category_mismatch}")
print(f"Region mismatches: {region_mismatch}")


Merge match status:
 _merge
both          100
left_only       0
right_only      0
Name: count, dtype: int64

Category mismatches: 0
Region mismatches: 0


Every store+product pair is present in both, and `sales_data`'s `Category`/`Region` match `sku_master` exactly. **`sales_data` is a trustworthy combined view** — we can rely on it later without treating it as a second source of truth that might disagree with the others.


### A genuine data quirk worth flagging: category isn't fixed per product

You might expect `Product ID` to always belong to one category (e.g. `P0001` is always "Electronics"). It isn't, in this dataset — the **same product is categorized differently depending on which store sells it**.


In [10]:
category_by_product = sku.pivot(index='Product ID', columns='Store ID', values='Category')
category_by_product


Store ID,S001,S002,S003,S004,S005
Product ID,,,,,
P0001,Electronics,Groceries,Toys,Groceries,Groceries
P0002,Clothing,Toys,Groceries,Groceries,Electronics
P0003,Clothing,Groceries,Electronics,Furniture,Toys
P0004,Electronics,Groceries,Groceries,Groceries,Groceries
P0005,Groceries,Electronics,Toys,Clothing,Groceries
P0006,Toys,Clothing,Toys,Groceries,Groceries
P0007,Groceries,Toys,Groceries,Groceries,Groceries
P0008,Electronics,Groceries,Clothing,Furniture,Groceries
P0009,Clothing,Groceries,Groceries,Clothing,Groceries


In [11]:
n_products_with_multiple_categories = (sku.groupby('Product ID')['Category'].nunique() > 1).sum()
print(f"Products whose category varies by store: {n_products_with_multiple_categories} out of {sku['Product ID'].nunique()}")


Products whose category varies by store: 20 out of 20


**This is not an error we're going to silently fix** — we already confirmed `sku_master` and `sales_data` agree with each other, so it's a consistent characteristic of the data, not a mismatch. But it *is* a modeling decision to flag going forward: **"category" only makes sense at the store+product level here, not at the product level alone.** Any category-based analysis or forecasting later needs to group by `(store_id, product_id)`, not `product_id` by itself. We leave the data as-is and simply carry this note forward.


## Step 7 — Standardize column names


In [12]:
def to_snake_case(col):
    return (col.strip().lower()
               .replace('/', '_')
               .replace(' ', '_')
               .replace('%', 'pct'))

for df in [sales, inventory, sku, calendar, sales_data]:
    df.columns = [to_snake_case(c) for c in df.columns]

print("sales:      ", list(sales.columns))
print("inventory:  ", list(inventory.columns))
print("sku_master: ", list(sku.columns))
print("calendar:   ", list(calendar.columns))
print("sales_data: ", list(sales_data.columns))


sales:       ['date', 'store_id', 'product_id', 'units_sold', 'demand', 'price', 'discount', 'promotion']
inventory:   ['date', 'store_id', 'product_id', 'inventory_level', 'units_ordered', 'competitor_pricing']
sku_master:  ['store_id', 'product_id', 'category', 'region']
calendar:    ['date', 'year', 'quarter', 'month', 'monthname', 'day', 'dayofweek', 'dayname', 'weekofyear', 'isweekend', 'seasonality', 'epidemic']
sales_data:  ['date', 'store_id', 'product_id', 'category', 'region', 'inventory_level', 'units_sold', 'units_ordered', 'price', 'discount', 'weather_condition', 'promotion', 'competitor_pricing', 'seasonality', 'epidemic', 'demand']


## Step 8 — Validate value ranges and business logic

Checking for negative values and values outside their expected ranges.


In [13]:
print("Negative value checks:")
for col in ['inventory_level', 'units_ordered', 'competitor_pricing']:
    print(f"  inventory.{col}: {(inventory[col] < 0).sum()} negative")
for col in ['units_sold', 'demand', 'price', 'discount']:
    print(f"  sales.{col}: {(sales[col] < 0).sum()} negative")

print("\nDiscount values:  ", sorted(sales['discount'].unique()))
print("Promotion values: ", sorted(sales['promotion'].unique()))
print("Epidemic values:  ", sorted(calendar['epidemic'].unique()))


Negative value checks:
  inventory.inventory_level: 0 negative
  inventory.units_ordered: 0 negative
  inventory.competitor_pricing: 0 negative
  sales.units_sold: 0 negative
  sales.demand: 0 negative
  sales.price: 0 negative
  sales.discount: 0 negative

Discount values:   [np.int64(0), np.int64(5), np.int64(10), np.int64(15), np.int64(20), np.int64(25)]


Promotion values:  [np.int64(0), np.int64(1)]
Epidemic values:   [np.int64(0), np.int64(1)]


No negative values anywhere, and all flag/categorical columns hold only their expected values.


### `units_sold` vs `inventory_level` — sanity check

We can never sell more units than were physically in stock that day. Let's confirm the data respects that.


In [14]:
daily_check = sales.merge(inventory, on=['date', 'store_id', 'product_id'])
impossible_sales = (daily_check['units_sold'] > daily_check['inventory_level']).sum()
print(f"Rows where units_sold > inventory_level (physically impossible): {impossible_sales}")


Rows where units_sold > inventory_level (physically impossible): 0


Zero — good, the data is internally consistent on this front.


### `demand` vs `units_sold` — what is `demand`, exactly?

Unlike `inventory_level`, `demand` is *allowed* to differ from `units_sold` in either direction — it looks like a forecast/estimate of underlying demand, not a hard cap. Let's check how often actual sales exceed the demand estimate (which would be normal for a forecast, but not for a physical limit).


In [15]:
sold_more_than_demand = (sales['units_sold'] > sales['demand']).sum()
print(f"Rows where units_sold > demand: {sold_more_than_demand} ({sold_more_than_demand/len(sales)*100:.1f}%)")
sales[['units_sold', 'demand']].describe()


Rows where units_sold > demand: 21000 (27.6%)


,units_sold,demand
count,76000.000000,76000.000000
mean,88.827316,104.317158
std,43.994525,46.964801
min,0.000000,4.000000
25%,58.000000,71.000000
50%,84.000000,100.000000
75%,114.000000,133.000000
max,426.000000,430.000000


About a quarter of rows sell *more* than the `demand` estimate suggested. That confirms `demand` behaves like a **forecast column**, not a ceiling — we leave it untouched, but note it as a candidate benchmark to beat later (same role `demand_forecast` played in the previous dataset).


## Step 9 — Outlier check: real signal or data error?

Same IQR method as before, but this time we expect (and want) to find real extremes, since NorthBay's actual problem is stockouts and overstock — those *should* show up as outliers.


In [16]:
def iqr_bounds(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

for col in ['inventory_level', 'units_ordered', 'competitor_pricing']:
    lo, hi = iqr_bounds(inventory[col])
    n_out = ((inventory[col] < lo) | (inventory[col] > hi)).sum()
    print(f"{col:20s} outliers={n_out:5d}  bounds=({lo:.1f}, {hi:.1f})  max={inventory[col].max()}")

print()
for col in ['units_sold', 'price', 'demand']:
    lo, hi = iqr_bounds(sales[col])
    n_out = ((sales[col] < lo) | (sales[col] > hi)).sum()
    print(f"{col:20s} outliers={n_out:5d}  bounds=({lo:.1f}, {hi:.1f})  max={sales[col].max()}")


inventory_level      outliers= 2759  bounds=(-272.0, 816.0)  max=2267
units_ordered        outliers= 7524  bounds=(-181.5, 302.5)  max=1616
competitor_pricing   outliers=  185  bounds=(-65.3, 195.9)  max=261.22



units_sold           outliers= 1411  bounds=(-26.0, 198.0)  max=426
price                outliers=   70  bounds=(-63.8, 191.6)  max=228.03
demand               outliers=  986  bounds=(-22.0, 226.0)  max=430


**`inventory_level` and `units_ordered` show large numbers of high-side outliers** — e.g. inventory up to 2267 units against a typical range of roughly 0–800. Unlike the flat, uniform dataset we saw last time, these look like **genuine overstock events**, exactly the kind of pattern this project is meant to catch. We leave them in the data untouched — removing them would erase the very signal we're trying to detect.


### Real stockouts, for the first time

This dataset actually has `inventory_level = 0` rows — unlike the previous version, where inventory never dropped below 50.


In [17]:
n_stockouts = (inventory['inventory_level'] == 0).sum()
print(f"Rows with inventory_level = 0: {n_stockouts} ({n_stockouts/len(inventory)*100:.2f}% of all rows)")

stockouts_by_product = inventory[inventory['inventory_level'] == 0].groupby('product_id').size().sort_values(ascending=False)
print("\nStockout counts by product (top 5):")
print(stockouts_by_product.head())

stockouts_by_store = inventory[inventory['inventory_level'] == 0].groupby('store_id').size()
print("\nStockout counts by store:")
print(stockouts_by_store)


Rows with inventory_level = 0: 406 (0.53% of all rows)

Stockout counts by product (top 5):
product_id
P0016    42
P0011    32
P0001    31
P0002    26
P0015    26
dtype: int64

Stockout counts by store:
store_id
S001    59
S002    93
S003    85
S004    78
S005    91
dtype: int64


A small but real fraction of rows (about 0.5%) are literal stockouts, concentrated more heavily in a handful of products (e.g. `P0016`, `P0011`, `P0001`). This is genuinely useful for the project — we now have **real historical stockout events** to learn from, rather than needing a proxy metric like we had to build for the previous dataset.


## Step 10 — Check relationships between numeric columns

A quick look at whether price, discount, and promotion actually move `units_sold` this time — worth checking since the previous dataset showed no relationship at all.


In [18]:
print("Correlation with units_sold:")
print("  price:     ", sales['units_sold'].corr(sales['price']).round(3))
print("  discount:  ", sales['units_sold'].corr(sales['discount']).round(3))
print("  promotion: ", sales['units_sold'].corr(sales['promotion']).round(3))

daily_pc = sales.merge(inventory, on=['date', 'store_id', 'product_id'])
print("\nCorrelation price vs competitor_pricing:", daily_pc['price'].corr(daily_pc['competitor_pricing']).round(3))


Correlation with units_sold:
  price:      -0.015
  discount:   0.184
  promotion:  0.227



Correlation price vs competitor_pricing: 0.977


Unlike the previous dataset, `discount` and `promotion` now show a real positive relationship with `units_sold` (~0.18 and ~0.23), and `price` tracks `competitor_pricing` very closely (~0.98). These are promising, usable signals for forecasting — we'll dig into them properly in the EDA notebook, but it's worth noting here since it changes how much value these columns will add later.


## Step 11 — Optimize data types


In [19]:
for df, cols in [(sales, ['store_id', 'product_id']),
                  (inventory, ['store_id', 'product_id']),
                  (sku, ['store_id', 'product_id', 'category', 'region']),
                  (calendar, ['monthname', 'dayname', 'seasonality']),
                  (sales_data, ['store_id', 'product_id', 'category', 'region', 'weather_condition', 'seasonality'])]:
    for c in cols:
        df[c] = df[c].astype('category')

sales['discount'] = sales['discount'].astype('int8')
sales['promotion'] = sales['promotion'].astype('int8')
calendar['epidemic'] = calendar['epidemic'].astype('int8')
sales_data['discount'] = sales_data['discount'].astype('int8')
sales_data['promotion'] = sales_data['promotion'].astype('int8')
sales_data['epidemic'] = sales_data['epidemic'].astype('int8')

print("Done. Example dtypes (sales):")
print(sales.dtypes)


Done. Example dtypes (sales):
date          datetime64[us]
store_id            category
product_id          category
units_sold             int64
demand                 int64
price                float64
discount                int8
promotion               int8
dtype: object


## Step 12 — Final verification


In [20]:
for name, df in [('sales', sales), ('inventory', inventory), ('sku', sku),
                  ('calendar', calendar), ('sales_data', sales_data)]:
    print(f"{name:12s} shape={df.shape}  missing={df.isnull().sum().sum()}  exact_dupes={df.duplicated().sum()}")


sales        shape=(76000, 8)  missing=0  exact_dupes=0
inventory    shape=(76000, 6)  missing=0  exact_dupes=0
sku          shape=(100, 4)  missing=0  exact_dupes=0


calendar     shape=(760, 12)  missing=0  exact_dupes=0


sales_data   shape=(76000, 16)  missing=0  exact_dupes=0


## Step 13 — Save the cleaned tables

Each table is saved separately, matching the original five files, with a `_cleaned` suffix so the raw uploads are never overwritten.


In [21]:
sales.to_csv('../Cleaned + Splitted/sales_daily_cleaned_v2.csv', index=False)
inventory.to_csv('../Cleaned + Splitted/inventory_snapshots_cleaned_v2.csv', index=False)
sku.to_csv('../Cleaned + Splitted/sku_master_cleaned_v2.csv', index=False)
calendar.to_csv('../Cleaned + Splitted/calender_cleaned_v2.csv', index=False)
sales_data.to_csv('../Cleaned + Splitted/sales_data_cleaned_v2.csv', index=False)

print("Saved 5 cleaned files to ../Cleaned + Splitted/ (with _v2 suffix, matching the actual files used downstream — previously this cell wrote non-suffixed filenames that nothing else reads).")


Saved 5 cleaned files to ../Cleaned + Splitted/ (with _v2 suffix, matching the actual files used downstream — previously this cell wrote non-suffixed filenames that nothing else reads).


## Summary — what we found and fixed

| Table | Issue found | Fix applied |
|---|---|---|
| all tables | `Date` as text; `sales_data` used a different format (`M/D/YYYY`) than the rest (`YYYY-MM-DD`) | Converted all to real `datetime64` |
| all tables | Column names with spaces | Renamed to `snake_case` |
| `sales_data` vs `sku_master` | Needed to verify the combined file actually agrees with the source tables | Verified — 100% match on category/region per store+product |
| `sku_master` | Same `product_id` maps to *different* categories depending on the store | **Left as-is** — confirmed consistent across files, documented as a modeling note (group by store+product, not product alone) |
| `inventory_level` / `units_ordered` | Large high-side outliers (inventory up to 2267) | **Left as-is** — genuine overstock signal, exactly what this project needs to detect |
| `inventory_level` | 0.53% of rows are literal stockouts (`inventory_level = 0`) | **Left as-is** — first dataset with real stockout events to learn from |
| `demand` vs `units_sold` | Sales exceed the `demand` estimate ~28% of the time | Confirmed `demand` is a forecast-style column, not a hard cap — no fix needed, noted as a benchmark |
| all tables | Missing values, duplicate rows, duplicate business keys, mismatched IDs | **Checked — none found** |

**New in this dataset compared to the last one:** real stockouts, real overstock outliers, and discount/promotion actually correlating with sales — all of which make this a much richer dataset for the demand-forecasting and risk-flagging objectives.

### Next step: EDA, tailored to these new findings.
